<a href="https://colab.research.google.com/github/thanhungoc2552006-debug/MoE-SmartHome/blob/main/notebooks/02_aruba_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path(
    "/content/drive/MyDrive/Study/Project_MoE/DATASET/aruba.txt"
)

assert DATA_PATH.exists(), f"Không tìm thấy: {DATA_PATH}"

EXPECTED_RAW_EVENTS = 1_719_558
EXPECTED_DUPLICATES = 6_425
EXPECTED_PSEUDO_EVENTS = 5

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
records = []
parser_issues = []

with DATA_PATH.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as file:
    for source_order, line in enumerate(file):
        parts = line.strip().split()

        if len(parts) < 4:
            parser_issues.append({
                "source_order": source_order,
                "reason": "fewer_than_4_fields",
                "raw_line": line.strip(),
            })
            continue

        date, time, sensor, value = parts[:4]
        activity = None
        marker = None

        if (
            len(parts) >= 6
            and parts[-1].lower() in {"begin", "end"}
        ):
            activity = " ".join(parts[4:-1])
            marker = parts[-1].lower()

        elif len(parts) != 4:
            parser_issues.append({
                "source_order": source_order,
                "reason": "unrecognized_extra_fields",
                "raw_line": line.strip(),
            })

        records.append({
            "source_order": source_order,
            "date": date,
            "time": time,
            "sensor": sensor,
            "value": value,
            "activity": activity,
            "marker": marker,
        })

raw_df = pd.DataFrame.from_records(records)
parser_issues_df = pd.DataFrame(parser_issues)

# Hỗ trợ timestamp có hoặc không có microsecond
raw_df["timestamp"] = pd.to_datetime(
    raw_df["date"] + " " + raw_df["time"],
    format="mixed",
    errors="coerce",
)

print("Raw events:", len(raw_df))
print("Parser issues:", len(parser_issues_df))
print("Invalid timestamps:", raw_df["timestamp"].isna().sum())
print(
    "Chronological before cleaning:",
    raw_df["timestamp"].is_monotonic_increasing,
)

assert len(raw_df) == EXPECTED_RAW_EVENTS
assert parser_issues_df.empty
assert raw_df["timestamp"].notna().all()

Raw events: 1719558
Parser issues: 0
Invalid timestamps: 0
Chronological before cleaning: False


In [8]:
# Các cột xác định nội dung event gốc
identity_columns = [
    "date",
    "time",
    "sensor",
    "value",
    "activity",
    "marker",
]

# keep="first": đánh dấu bản sao thứ hai cần loại
duplicate_mask = raw_df.duplicated(
    subset=identity_columns,
    keep="first",
)

duplicate_positions = np.flatnonzero(
    duplicate_mask.to_numpy()
)

n_duplicates = len(duplicate_positions)

assert n_duplicates == EXPECTED_DUPLICATES
assert np.all(np.diff(duplicate_positions) == 1)

# Xác minh block thứ hai giống hoàn toàn block ngay trước nó
block_size = n_duplicates
second_start = duplicate_positions[0]
first_start = second_start - block_size

first_block = (
    raw_df.iloc[first_start:second_start][identity_columns]
    .reset_index(drop=True)
)

second_block = (
    raw_df.iloc[
        second_start:second_start + block_size
    ][identity_columns]
    .reset_index(drop=True)
)

blocks_identical = first_block.equals(second_block)

assert blocks_identical, (
    "6.425 duplicate không tạo thành block lặp đã xác nhận"
)

# Lưu audit log chi tiết trước khi loại
removal_log_df = raw_df.loc[
    duplicate_mask,
    [
        "source_order",
        "timestamp",
        "sensor",
        "value",
        "activity",
        "marker",
    ],
].copy()

removal_log_df["duplicate_of_source_order"] = (
    raw_df.iloc[first_start:second_start][
        "source_order"
    ].to_numpy()
)

removal_log_df["removal_reason"] = (
    "verified_contiguous_duplicate_block"
)

# Stream sạch, vẫn giữ pseudo-events
events_df = (
    raw_df.loc[~duplicate_mask]
    .copy()
    .reset_index(drop=True)
)

# Phân loại physical sensors và pseudo-events
sensor_text = (
    events_df["sensor"]
    .astype("string")
    .str.strip()
    .str.upper()
)

physical_mask = sensor_text.str.fullmatch(
    r"[MDT]\d{3}",
    na=False,
)

pseudo_mask = sensor_text.isin([
    "LEAVEHOME",
    "ENTERHOME",
])

unknown_mask = ~(physical_mask | pseudo_mask)

events_df["event_kind"] = "unknown"
events_df.loc[physical_mask, "event_kind"] = (
    "physical_sensor"
)
events_df.loc[pseudo_mask, "event_kind"] = (
    "pseudo_event"
)

events_df["sensor_type"] = pd.Series(
    pd.NA,
    index=events_df.index,
    dtype="string",
)

events_df.loc[physical_mask, "sensor_type"] = (
    sensor_text.loc[physical_mask].str[0]
)

events_df.loc[pseudo_mask, "sensor_type"] = "PSEUDO"

# Pseudo-events được giữ để audit nhưng không dùng làm feature
events_df["use_as_model_feature"] = physical_mask

physical_events_df = (
    events_df.loc[physical_mask]
    .copy()
    .reset_index(drop=True)
)

pseudo_events_df = (
    events_df.loc[pseudo_mask]
    .copy()
    .reset_index(drop=True)
)

unknown_events_df = (
    events_df.loc[unknown_mask]
    .copy()
)

audit_summary_df = pd.DataFrame({
    "metric": [
        "raw_events",
        "duplicates_removed",
        "clean_events",
        "physical_sensor_events",
        "pseudo_events",
        "unknown_events",
        "invalid_timestamps",
        "chronological_after_cleaning",
    ],
    "value": [
        len(raw_df),
        len(removal_log_df),
        len(events_df),
        len(physical_events_df),
        len(pseudo_events_df),
        len(unknown_events_df),
        int(events_df["timestamp"].isna().sum()),
        events_df["timestamp"].is_monotonic_increasing,
    ],
})

display(audit_summary_df)
display(pseudo_events_df)
display(removal_log_df.head())

assert len(events_df) == 1_713_133
assert len(physical_events_df) == 1_713_128
assert len(pseudo_events_df) == EXPECTED_PSEUDO_EVENTS
assert unknown_events_df.empty
assert events_df["timestamp"].notna().all()
assert events_df["timestamp"].is_monotonic_increasing
assert not physical_events_df["sensor"].isin(
    ["LEAVEHOME", "ENTERHOME"]
).any()

,metric,value
0,raw_events,1719558
1,duplicates_removed,6425
2,clean_events,1713133
3,physical_sensor_events,1713128
4,pseudo_events,5
5,unknown_events,0
6,invalid_timestamps,0
7,chronological_after_cleaning,True


,source_order,date,time,sensor,value,activity,marker,timestamp,event_kind,sensor_type,use_as_model_feature
0,1523044,2011-05-17,11:40:03.013619,LEAVEHOME,180,None,None,2011-05-17 11:40:03.013619,pseudo_event,PSEUDO,False
1,1523584,2011-05-17,14:58:04.907136,LEAVEHOME,300,None,None,2011-05-17 14:58:04.907136,pseudo_event,PSEUDO,False
2,1526729,2011-05-17,18:31:56.044148,LEAVEHOME,300,None,None,2011-05-17 18:31:56.044148,pseudo_event,PSEUDO,False
3,1530032,2011-05-18,12:22:23.315366,LEAVEHOME,300,None,None,2011-05-18 12:22:23.315366,pseudo_event,PSEUDO,False
4,1530061,2011-05-18,14:12:13.738946,ENTERHOME,6592,None,None,2011-05-18 14:12:13.738946,pseudo_event,PSEUDO,False


,source_order,timestamp,sensor,value,activity,marker,duplicate_of_source_order,removal_reason
1571287,1571287,2011-05-23 00:02:56.715882,T003,23,None,None,1564862,verified_contiguous_duplicate_block
1571288,1571288,2011-05-23 00:18:07.424272,T005,22,None,None,1564863,verified_contiguous_duplicate_block
1571289,1571289,2011-05-23 00:28:14.510947,T001,22,None,None,1564864,verified_contiguous_duplicate_block
1571290,1571290,2011-05-23 00:28:14.633770,T004,22.5,None,None,1564865,verified_contiguous_duplicate_block
1571291,1571291,2011-05-23 00:38:21.655845,T002,23,None,None,1564866,verified_contiguous_duplicate_block


In [10]:
# TASK 2.2 — ACTIVITY EPISODE RECONSTRUCTION
# Hỗ trợ nhiều activity khác tên cùng active

from collections import Counter
import numpy as np
import pandas as pd

required_columns = {
    "timestamp",
    "source_order",
    "activity",
    "marker",
}

missing_columns = required_columns - set(events_df.columns)

assert not missing_columns, (
    f"events_df thiếu các cột: {missing_columns}"
)

assert events_df["timestamp"].is_monotonic_increasing, (
    "events_df chưa đúng thứ tự thời gian"
)

# Chỉ đọc marker; không sửa events_df
marker_events_df = events_df.loc[
    events_df["marker"].isin(["begin", "end"]),
    [
        "timestamp",
        "source_order",
        "activity",
        "marker",
    ],
].copy()

assert marker_events_df["activity"].notna().all()
assert marker_events_df["timestamp"].is_monotonic_increasing


# --------------------------------------------------
# Validation setup
# --------------------------------------------------

error_types = [
    "same_activity_begin_while_active",
    "end_without_matching_begin",
    "activity_still_open_at_eof",
    "duration_not_positive",
]

validation_counts = Counter({
    error_type: 0
    for error_type in error_types
})

validation_errors = []


def add_validation_error(
    error_type,
    timestamp,
    source_order,
    activity,
    details=None,
):
    validation_counts[error_type] += 1

    validation_errors.append({
        "error_type": error_type,
        "timestamp": timestamp,
        "source_order": source_order,
        "activity": activity,
        "details": details,
    })


# --------------------------------------------------
# State machine
# --------------------------------------------------

# Key: activity name
# Value: begin-marker information
active_activities = {}

episodes = []
max_simultaneously_active = 0

for row in marker_events_df.itertuples(index=False):
    activity = row.activity
    marker = row.marker

    # ----------------------------------------------
    # BEGIN
    # ----------------------------------------------
    if marker == "begin":
        if activity in active_activities:
            add_validation_error(
                error_type=(
                    "same_activity_begin_while_active"
                ),
                timestamp=row.timestamp,
                source_order=row.source_order,
                activity=activity,
                details=(
                    "Duplicate begin ignored; original "
                    "begin remains active."
                ),
            )
            continue

        # Activity khác tên có thể overlap hợp lệ
        active_activities[activity] = {
            "start_time": row.timestamp,
            "start_source_order": row.source_order,
        }

        max_simultaneously_active = max(
            max_simultaneously_active,
            len(active_activities),
        )

    # ----------------------------------------------
    # END
    # ----------------------------------------------
    elif marker == "end":
        if activity not in active_activities:
            add_validation_error(
                error_type="end_without_matching_begin",
                timestamp=row.timestamp,
                source_order=row.source_order,
                activity=activity,
                details=(
                    "No active begin marker for this "
                    "activity name."
                ),
            )
            continue

        begin_info = active_activities.pop(activity)

        duration_seconds = (
            row.timestamp
            - begin_info["start_time"]
        ).total_seconds()

        if duration_seconds <= 0:
            add_validation_error(
                error_type="duration_not_positive",
                timestamp=row.timestamp,
                source_order=row.source_order,
                activity=activity,
                details=(
                    f"duration_seconds={duration_seconds}"
                ),
            )
            continue

        episodes.append({
            "activity": activity,
            "start_time": begin_info["start_time"],
            "end_time": row.timestamp,
            "start_source_order": (
                begin_info["start_source_order"]
            ),
            "end_source_order": row.source_order,
            "duration_seconds": duration_seconds,
        })


# --------------------------------------------------
# Activities còn mở ở cuối dataset
# --------------------------------------------------

for activity, begin_info in active_activities.items():
    add_validation_error(
        error_type="activity_still_open_at_eof",
        timestamp=begin_info["start_time"],
        source_order=begin_info["start_source_order"],
        activity=activity,
        details="Dataset ended before matching end marker.",
    )


# --------------------------------------------------
# Tạo episodes_df
# --------------------------------------------------

episodes_df = (
    pd.DataFrame(episodes)
    .sort_values(
        ["start_time", "start_source_order"],
        kind="stable",
    )
    .reset_index(drop=True)
)

episodes_df.insert(
    0,
    "episode_id",
    np.arange(len(episodes_df)),
)

episode_columns = [
    "episode_id",
    "activity",
    "start_time",
    "end_time",
    "start_source_order",
    "end_source_order",
    "duration_seconds",
]

episodes_df = episodes_df[episode_columns]


# --------------------------------------------------
# Đếm episode overlap với ít nhất một episode khác
# Hai episode chỉ chạm biên không được tính là overlap
# --------------------------------------------------

overlap_flags = np.zeros(
    len(episodes_df),
    dtype=bool,
)

max_end_time = None
max_end_episode_index = None

for episode_index, row in episodes_df.iterrows():
    if (
        max_end_time is not None
        and row["start_time"] < max_end_time
    ):
        overlap_flags[episode_index] = True
        overlap_flags[max_end_episode_index] = True

    if (
        max_end_time is None
        or row["end_time"] > max_end_time
    ):
        max_end_time = row["end_time"]
        max_end_episode_index = episode_index

n_overlapping_episodes = int(overlap_flags.sum())


# --------------------------------------------------
# Reports
# --------------------------------------------------

validation_summary_df = pd.DataFrame({
    "error_type": error_types,
    "count": [
        validation_counts[error_type]
        for error_type in error_types
    ],
})

validation_errors_df = pd.DataFrame(
    validation_errors
)

activity_episode_counts_df = (
    episodes_df["activity"]
    .value_counts()
    .rename_axis("activity")
    .reset_index(name="episode_count")
)

print(
    "Total reconstructed episodes:",
    len(episodes_df),
)

print(
    "Maximum simultaneously active activities:",
    max_simultaneously_active,
)

print(
    "Episodes overlapping another episode:",
    n_overlapping_episodes,
)

print("\nValidation errors:")
display(validation_summary_df)

print("\nFirst 10 episodes:")
display(episodes_df.head(10))

print("\nActivity episode counts:")
display(activity_episode_counts_df)

if not validation_errors_df.empty:
    print("\nValidation error details:")
    display(validation_errors_df)

Total reconstructed episodes: 6440
Maximum simultaneously active activities: 2
Episodes overlapping another episode: 91

Validation errors:


,error_type,count
0,same_activity_begin_while_active,0
1,end_without_matching_begin,0
2,activity_still_open_at_eof,0
3,duration_not_positive,0



First 10 episodes:


,episode_id,activity,start_time,end_time,start_source_order,end_source_order,duration_seconds
0,0,Sleeping,2010-11-04 00:03:50.209589,2010-11-04 05:40:43.642664,0,48,20213.433075
1,1,Bed_to_Toilet,2010-11-04 05:40:51.303739,2010-11-04 05:43:30.279021,52,63,158.975282
2,2,Sleeping,2010-11-04 05:43:45.732400,2010-11-04 08:01:12.282970,68,172,8246.550570
3,3,Meal_Preparation,2010-11-04 08:11:09.966157,2010-11-04 08:27:02.801314,266,520,952.835157
4,4,Meal_Preparation,2010-11-04 08:33:52.929406,2010-11-04 08:35:45.822482,659,707,112.893076
5,5,Relax,2010-11-04 09:29:23.223133,2010-11-04 09:34:05.780570,934,953,282.557437
6,6,Housekeeping,2010-11-04 09:34:16.851472,2010-11-04 09:44:40.315639,962,1047,623.464167
7,7,Meal_Preparation,2010-11-04 09:48:52.342223,2010-11-04 09:53:02.954445,1129,1228,250.612222
8,8,Meal_Preparation,2010-11-04 09:54:58.753743,2010-11-04 09:56:27.334395,1261,1307,88.580652
9,9,Eating,2010-11-04 09:56:41.831135,2010-11-04 09:59:04.438334,1318,1339,142.607199



Activity episode counts:


,activity,episode_count
0,Relax,2907
1,Meal_Preparation,1596
2,Leave_Home,427
3,Enter_Home,427
4,Sleeping,398
5,Eating,255
6,Work,171
7,Bed_to_Toilet,156
8,Wash_Dishes,64
9,Housekeeping,33
